# Day 1: Credit Card Fraud Exploratory Data Analysis & SQL Profiling

**Project**: Explainable Credit Card Fraud Detection & Risk Scoring  
**Dataset**: Kaggle ULB Credit Card Fraud Detection (284,807 transactions)  
**Goal**: Comprehensive data validation, statistical distribution profiling, DuckDB SQL analytics, and interpretability-oriented visualization.

## 1. Setup & Imports
We import core data science libraries and the reusable modular functions authored under `src/`.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import duckdb

from src.ingestion.data_loader import load_raw_data, validate_dataset
from src.ingestion.sql_analytics import run_all_sql_analytics, get_duckdb_connection
from src.features.eda_analysis import (
    compute_class_distribution,
    compute_amount_comparison,
    compute_correlations_with_target,
    compute_amount_bands_summary,
)

pd.set_option('display.max_columns', 35)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print('Environment initialized successfully.')

## 2. Data Ingestion & Integrity Validation
We load `data/raw/creditcard.csv` and validate dimensions, types, missing values, duplicates, and financial constraints.

In [ ]:
DATA_PATH = ROOT_DIR / 'data' / 'raw' / 'creditcard.csv'
df = load_raw_data(DATA_PATH)
report = validate_dataset(df)

print(f'Dataset Valid: {report.is_valid}')
print(f'Rows: {report.total_rows:,} | Columns: {report.total_cols}')
print(f'Missing Values: {report.missing_values_count}')
print(f'Duplicate Rows: {report.duplicate_rows_count:,} (Fraud Duplicates: {report.duplicate_fraud_count}, Legit Duplicates: {report.duplicate_legit_count:,})')
print(f'Zero-Dollar Authorizations: {report.zero_amount_count:,} (Zero-Dollar Fraud: {report.zero_amount_fraud_count})')
df.head()

## 3. Class Imbalance Analysis
Evaluating the extreme class imbalance ratio (~578:1) between non-fraud and fraudulent transactions.

In [ ]:
dist = compute_class_distribution(df)
print(f"Legitimate Transactions: {dist['legit_count']:,} ({dist['legit_percentage']:.3f}%)")
print(f"Fraudulent Transactions: {dist['fraud_count']:,} ({dist['fraud_percentage']:.3f}%)")
print(f"Imbalance Ratio: {dist['imbalance_ratio']}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].bar(['Legitimate', 'Fraudulent'], [dist['legit_count'], dist['fraud_count']], color=['#2b5c8f', '#d9534f'], edgecolor='black')
ax[0].set_yscale('log')
ax[0].set_title('Transaction Counts (Log Scale)')
ax[0].grid(axis='y', linestyle='--', alpha=0.5)

ax[1].bar(['Legitimate', 'Fraudulent'], [dist['legit_percentage'], dist['fraud_percentage']], color=['#2b5c8f', '#d9534f'], edgecolor='black')
ax[1].set_title('Transaction Proportions (%)')
ax[1].grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Transaction Amount Distribution & Skewness
Analyzing the differences in transaction amounts between legitimate and fraudulent behaviors.

In [ ]:
amt_comp = compute_amount_comparison(df)
display(amt_comp)

fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot([np.log1p(df[df['Class'] == 0]['Amount']), np.log1p(df[df['Class'] == 1]['Amount'])],
           labels=['Legitimate', 'Fraudulent'], patch_artist=True,
           boxprops=dict(facecolor='#cbe3f2', color='#2b5c8f'),
           medianprops=dict(color='#d9534f', linewidth=2))
ax.set_ylabel('Log(Amount + $1)')
ax.set_title('Distribution of Log(Amount + 1) by Class')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 5. High-Performance SQL Analytics via DuckDB
DuckDB provides vectorized analytical SQL queries to profile transaction volume, financial amounts, and time buckets.

In [ ]:
con = get_duckdb_connection(df)
sql_results = run_all_sql_analytics(df)

print('=== Transaction Distribution & Total Volume ===')
display(sql_results['class_counts'])

print('\n=== Amount Descriptive Statistics by Class ===')
display(sql_results['amount_stats'])

print('\n=== Fraud Rate by Amount Bracket ===')
display(sql_results['amount_tiers'])

## 6. Feature Correlation & Discriminatory PCA Components
Identifying the PCA anonymized components exhibiting the strongest positive and negative correlations with fraudulent activity.

In [ ]:
corr = compute_correlations_with_target(df)
top_neg = corr.head(6)
top_pos = corr.tail(6)
top_corr = pd.concat([top_neg, top_pos])

plt.figure(figsize=(10, 5))
colors = ['#d9534f' if v > 0 else '#2b5c8f' for v in top_corr.values]
plt.barh(top_corr.index, top_corr.values, color=colors, edgecolor='black', alpha=0.85)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 12 Features Most Correlated with Fraud Target (Class)')
plt.xlabel('Pearson Correlation Coefficient')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Key Findings & Next Steps (Day 2/3 Blueprint)
1. **Imbalance**: 0.173% fraud rate (~578:1 ratio) dictates PR-AUC evaluation and cost-sensitive threshold tuning.
2. **Duplicates**: 1,081 duplicates contain 19 fraud cases; dropping them would cause positive class signal loss.
3. **Zero-Dollar Authorizations**: 1,825 transactions are $0.00 with a 1.48% fraud rate (8.6x baseline); create an `is_zero_auth` flag.
4. **Key Discriminators**: V14, V17, V12, V10 (negative correlation) and V4, V11 (positive correlation) show distinct separation.
5. **Temporal Cycle**: Off-peak night hours exhibit higher fraud frequency despite lower total transaction volume.